# IMC Prosperity Manual Challenge Round 4: Vanilla Just Isn't Exotic Enough

This notebook prices the visible Round 4 AETHER_CRYSTAL contracts under the stated GBM model and compares fair values to bid/ask quotes.

Assumptions:
- Spot `S0 = 50`, inferred from the underlying quote.
- Annualized volatility `sigma = 251%`.
- Zero risk-neutral drift.
- 252 trading days/year, 4 simulation steps/day.
- 2 weeks = 10 trading days = 40 steps.
- 3 weeks = 15 trading days = 60 steps.
- Contract size = 3000, applied as a final PnL multiplier.
- Binary put assumed to pay `10` if `S_T < 40`, based on the quote scale.
- Knock-out put assumed to be strike `45`, barrier `35`, based on the observed quote scale.


In [ ]:
import numpy as np
import pandas as pd
from math import log, sqrt, exp, erf

S0 = 50.0
SIGMA = 2.51
TRADING_DAYS_PER_YEAR = 252
STEPS_PER_DAY = 4
CONTRACT_SIZE = 3000

def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def years_from_trading_days(days):
    return days / TRADING_DAYS_PER_YEAR

def bs_call(S, K, T, sigma):
    d1 = (log(S / K) + 0.5 * sigma**2 * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * norm_cdf(d1) - K * norm_cdf(d2)

def bs_put(S, K, T, sigma):
    d1 = (log(S / K) + 0.5 * sigma**2 * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return K * norm_cdf(-d2) - S * norm_cdf(-d1)

T_2W = years_from_trading_days(10)
T_3W = years_from_trading_days(15)


In [ ]:
# Monte Carlo for path-dependent exotics
N = 1_000_000
rng = np.random.default_rng(42)

steps_3w = 15 * STEPS_PER_DAY
choice_step = 10 * STEPS_PER_DAY
dt = 1 / (TRADING_DAYS_PER_YEAR * STEPS_PER_DAY)

Z = rng.standard_normal((N, steps_3w))
log_paths = np.log(S0) + np.cumsum((-0.5 * SIGMA**2) * dt + SIGMA * np.sqrt(dt) * Z, axis=1)
paths = np.exp(log_paths)

S_choice = paths[:, choice_step - 1]
S_final = paths[:, -1]
path_min = paths.min(axis=1)

chooser_payoff = np.where(
    S_choice >= 50,
    np.maximum(S_final - 50, 0),
    np.maximum(50 - S_final, 0),
)

binary_put_payoff = 10.0 * (S_final < 40)
ko_put_payoff = np.where(path_min < 35, 0, np.maximum(45 - S_final, 0))

fair_exotics = {
    "AC_50_CO": chooser_payoff.mean(),
    "AC_40_BP": binary_put_payoff.mean(),
    "AC_45_KO": ko_put_payoff.mean(),
}

fair_exotics


In [ ]:
contracts = [
    ("AC", "underlying", None, None, 49.975, 50.025, 200, 200),
    ("AC_50_P", "put", 50, 15, 12.00, 12.05, 50, 50),
    ("AC_50_C", "call", 50, 15, 12.00, 12.05, 50, 50),
    ("AC_35_P", "put", 35, 15, 4.33, 4.35, 50, 50),
    ("AC_40_P", "put", 40, 15, 6.50, 6.55, 50, 50),
    ("AC_45_P", "put", 45, 15, 9.05, 9.10, 50, 50),
    ("AC_60_C", "call", 60, 15, 8.80, 8.85, 50, 50),
    ("AC_50_P_2", "put", 50, 10, 9.70, 9.75, 50, 50),
    ("AC_50_C_2", "call", 50, 10, 9.70, 9.75, 50, 50),
    ("AC_50_CO", "chooser", 50, 15, 22.20, 22.30, 50, 50),
    ("AC_40_BP", "binary_put", 40, 15, 5.00, 5.10, 50, 50),
    ("AC_45_KO", "ko_put", 45, 15, 0.15, 0.175, 500, 500),
]

rows = []
for name, typ, K, days, bid, ask, bid_size, ask_size in contracts:
    if typ == "underlying":
        fair = S0
    elif typ == "call":
        fair = bs_call(S0, K, years_from_trading_days(days), SIGMA)
    elif typ == "put":
        fair = bs_put(S0, K, years_from_trading_days(days), SIGMA)
    else:
        fair = fair_exotics[name]

    buy_edge = fair - ask
    sell_edge = bid - fair

    if buy_edge > max(sell_edge, 0):
        action = "BUY"
        volume = ask_size
        unit_edge = buy_edge
    elif sell_edge > max(buy_edge, 0):
        action = "SELL"
        volume = bid_size
        unit_edge = sell_edge
    else:
        action = "NO TRADE"
        volume = 0
        unit_edge = 0.0

    rows.append({
        "contract": name,
        "type": typ,
        "fair": fair,
        "bid": bid,
        "ask": ask,
        "best_action": action,
        "volume": volume,
        "unit_edge": unit_edge,
        "expected_pnl_before_multiplier": unit_edge * volume,
        "expected_pnl_after_3000x": unit_edge * volume * CONTRACT_SIZE,
    })

df = pd.DataFrame(rows)
df.round(4)


In [ ]:
orders = df[df["best_action"] != "NO TRADE"][[
    "contract", "best_action", "volume", "fair", "bid", "ask", "unit_edge", "expected_pnl_after_3000x"
]].copy()

orders.round(4)


## Recommended orders

Given the assumptions above, the max-EV order set is:

| Contract | Side | Volume |
|---|---:|---:|
| AC_50_P_2 | Buy | 50 |
| AC_50_C_2 | Buy | 50 |
| AC_50_CO | Sell | 50 |
| AC_40_BP | Sell | 50 |
| AC_45_KO | Buy | 500 |

Do not trade the underlying or the near-fair 3-week vanilla options unless you want extra hedging for score variance. The highest-confidence edge is the 2-week ATM straddle, followed by selling the chooser. The binary put and knock-out put depend on payoff/barrier interpretation, so treat those as slightly lower confidence.
